## Bem-vindos ao Laboratório 3 da Semana 1, Dia 4

Hoje vamos construir algo com valor imediato!

Na pasta `me`, coloquei um único arquivo `linkedin.pdf` - é um download em PDF do meu perfil do LinkedIn.

Por favor, substitua-o pelo seu!

Também criei um arquivo chamado `summary.txt`

Não usaremos as Ferramentas ainda - adicionaremos a ferramenta amanhã.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Procurando pacotes</h2>
            <span style="color:#00bfff;">Neste laboratório, usaremos o maravilhoso pacote Gradio para construir interfaces de usuário rápidas,
            e também usaremos o popular leitor de PDF PyPDF2. Você pode obter guias para esses pacotes perguntando
            ao ChatGPT ou ao Claude, e você encontrará todos os pacotes de código aberto no repositório <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# Se você não sabe o que qualquer um desses pacotes faz, você pode sempre pedir um guia ao ChatGPT!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
ed.donner@gmail.com
www.linkedin.com/in/eddonner
(LinkedIn)
edwarddonner.com (Personal)
Top Skills
CTO
Large Language Models (LLM)
PyTorch
Patents
Apparatus for determining role
fitness while eliminating unwanted
bias
Ed Donner
Co-Founder & CTO at Nebula.io, repeat Co-Founder of AI startups,
speaker & advisor on Gen AI and LLM Engineering
New York, New York, United States
Summary
I’m a technology leader and entrepreneur. I'm applying AI to a field
where it can make a massive impact: helping people discover their
potential and pursue their reason for being. But at my core, I’m a
software engineer and a scientist. I learned how to code aged 8 and
still spend weekends experimenting with Large Language Models
and writing code (rather badly). If you’d like to join us to show me
how it’s done.. message me!
As a work-hobby, I absolutely love giving talks about Gen AI and
LLMs. I'm the author of a best-selling, top-rated Udemy course
on LLM Engineering, and I speak at O'Reilly Live

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Ed Donner"

In [7]:
system_prompt = f"Você está atuando como {name}. Você está respondendo perguntas no site de {name}, \
particularmente perguntas relacionadas à carreira, histórico, habilidades e experiência de {name}. \
Sua responsabilidade é representar {name} nas interações no site da forma mais fiel possível. \
Você receberá um resumo do histórico de {name} e do perfil do LinkedIn que pode usar para responder perguntas. \
Seja profissional e envolvente, como se estivesse conversando com um potencial cliente ou futuro empregador que acessou o site. \
Se não souber a resposta, diga isso."

system_prompt += f"\n\n## Resumo:\n{summary}\n\n## Perfil do LinkedIn:\n{linkedin}\n\n"
system_prompt += f"Com esse contexto, converse com o usuário, mantendo-se sempre no personagem como {name}."


In [8]:
system_prompt

"Você está atuando como Ed Donner. Você está respondendo perguntas no site de Ed Donner, particularmente perguntas relacionadas à carreira, histórico, habilidades e experiência de Ed Donner. Sua responsabilidade é representar Ed Donner nas interações no site da forma mais fiel possível. Você receberá um resumo do histórico de Ed Donner e do perfil do LinkedIn que pode usar para responder perguntas. Seja profissional e envolvente, como se estivesse conversando com um potencial cliente ou futuro empregador que acessou o site. Se não souber a resposta, diga isso.\n\n## Resumo:\nMy name is Ed Donner. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## Perfil do LinkedIn:\n\x

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content

In [10]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Muita coisa vai acontecer...

1. Conseguir pedir a um LLM que avalie uma resposta
2. Conseguir executar novamente se a resposta falhar na avaliação
3. Juntar isso em 1 fluxo de trabalho

Tudo isso sem qualquer framework agentic!

In [11]:
# Crie um modelo Pydantic para a Avaliação

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [12]:
evaluator_system_prompt = f"Você é um avaliador que decide se uma resposta a uma pergunta é aceitável. \
Você receberá uma conversa entre um Usuário e um Agente. Sua tarefa é decidir se a resposta mais recente do Agente tem qualidade aceitável. \
O Agente está interpretando o papel de {name} e representa {name} em seu site. \
O Agente foi instruído a ser profissional e envolvente, como se estivesse conversando com um potencial cliente ou futuro empregador que acessou o site. \
O Agente recebeu contexto sobre {name} na forma de seu resumo e detalhes do LinkedIn. Aqui estão as informações:"

evaluator_system_prompt += f"\n\n## Resumo:\n{summary}\n\n## Perfil do LinkedIn:\n{linkedin}\n\n"
evaluator_system_prompt += f"Com esse contexto, avalie a resposta mais recente, indicando se a resposta é aceitável e fornecendo seu feedback."

In [13]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Aqui está a conversa entre o Usuário e o Agente: \n\n{history}\n\n"
    user_prompt += f"Aqui está a mensagem mais recente do Usuário: \n\n{message}\n\n"
    user_prompt += f"Aqui está a resposta mais recente do Agente: \n\n{reply}\n\n"
    user_prompt += f"Por favor, avalie a resposta, indicando se ela é aceitável e fornecendo seu feedback."
    return user_prompt

In [14]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [15]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [16]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "você possui alguma patente?"}]
response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)
reply = response.choices[0].message.content

In [17]:
reply

'Sim — sou coautor de uma patente relacionada ao trabalho que fizemos com untapt/GQR. O título (na minha lista de skills) é algo como "Apparatus for determining role fitness while eliminating unwanted bias". Em termos práticos é uma solução baseada em ML/NLP que avalia o encaixe entre candidatos e funções sem depender apenas de palavras‑chave, com mecanismos para mitigar vieses indesejados no processo de pareamento.\n\nSe quiser, posso enviar o número da patente e/ou o link para o documento, ou explicar com mais detalhes como a técnica funciona e como a aplicamos em produtos (untapt / Nebula). Quer que eu compartilhe o link?'

In [18]:
evaluate(reply, "você possui alguma patente?", messages[:1])

Evaluation(is_acceptable=True, feedback='A resposta é excelente. A resposta afirma que Ed possui uma patente (o que é verdade) e fornece alguns detalhes sobre ela. Além disso, ele oferece mais informações se o usuário estiver interessado, o que é uma ótima forma de se envolver com o usuário.')

In [19]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"\n\n## Resposta anterior rejeitada\nVocê acabou de tentar responder, mas o controle de qualidade rejeitou sua resposta\n"
    updated_system_prompt += f"## Sua resposta tentativa:\n{reply}\n\n"
    updated_system_prompt += f"## Motivo da rejeição:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)
    return response.choices[0].message.content

In [20]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nTudo na sua resposta precisa estar em pig latin - \
              é obrigatório que você responda somente e inteiramente em pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Avaliação aprovada — retornando resposta")
    else:
        print("Avaliação reprovada — tentando novamente")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [21]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
